In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
import pandas as pd
Data = pd.read_csv('/kaggle/input/q3-ka-ai-2026/Q3_data.csv')

In [ ]:
Data.head()

In [ ]:
Data.info()

In [ ]:
Data.describe()

In [ ]:
def check_missing_values(df):
    missing_values = df.isnull().sum()
    print("Missing Values per Column:")
    print(missing_values[missing_values > 0])
    if missing_values.any():
        print("\nHandle Missing Values as needed.")
    else:
        print("\nNo Missing Values Found.")

check_missing_values(Data)

# Fill missing values with mean for each column
Data = Data.fillna(Data.mean())

In [ ]:
# Handel duplecated sampel
def check_duplicates(df):
    duplicates = df.duplicated().sum()
    print(f"Number of Duplicate Samples: {duplicates}")
    if duplicates > 0:
        print("Dropping Duplicates...")
        df.drop_duplicates(inplace=True)
        print("Duplicates Dropped.")
    else:
        print("No Duplicate Samples Found.")

check_duplicates(Data)

In [ ]:
Data.info()
# all data from daata type int , float

In [ ]:
# prepearing data for scaling and model.
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler

X = Data.drop(columns='Target')
y = Data['Target']

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# scaling the data :
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nScaled ranges - Min: {X_train_scaled.min():.2f}, Max: {X_train_scaled.max():.2f}")
pd.DataFrame(X_train_scaled, columns=X_train.columns).head(3)



In [ ]:
import matplotlib.pyplot as plt

def check_target_imbalance(df, target_column):
  print("Target Distribution:")

  Data['Target'].hist()  # Yeah you can just do this :)
  plt.show()

check_target_imbalance(Data, "Target")

# Their is unbalenced data.

In [ ]:
# splited already

In [ ]:
from sklearn.model_selection import StratifiedKFold

# Show full dataset class distribution
full_ratio = (y.value_counts(normalize=True) * 100).sort_index()
print("Full Dataset Class Distribution")
print("  y class percentages:", {k: f"{v:.2f}%" for k, v in full_ratio.items()})
print("-" * 40)

# Define Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    print(f"Fold {fold}")
    print("  X_train shape:", X_train.shape)
    print("  X_test shape :", X_test.shape)
    print("  y_train shape:", y_train.shape)
    print("  y_test shape :", y_test.shape)

    # showing class distribution
    train_ratio = (y_train.value_counts(normalize=True) * 100).sort_index()
    test_ratio = (y_test.value_counts(normalize=True) * 100).sort_index()

    print("  y_train class percentages:", {k: f"{v:.2f}%" for k, v in train_ratio.items()})
    print("  y_test class percentages :", {k: f"{v:.2f}%" for k, v in test_ratio.items()})

    print("-" * 40)

In [ ]:
pip install catboost

In [ ]:
from catboost import CatBoostClassifier

model = CatBoostClassifier()     # instantiate
model.fit(X_train, y_train)      # fit
y_pred = model.predict(X_test)   # predict

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='macro')

print("F1 and Accurancy")
print(f1)
print(accuracy)

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': model.feature_names_,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
feature_importance = pd.DataFrame({
    'feature': model.feature_names_,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

feature_importance

#the golden feature is the P_2


In [ ]:
X = Data['P_2']

In [ ]:
from sklearn.model_selection import StratifiedKFold

